## RANO

## Merge all the GTV in one dir

In [32]:
from pathlib import Path
from tqdm import tqdm
import nibabel as nib
import shutil
import numpy as np

source_dir = Path('./CFB-GBM/data')
target_dir = Path('./full_gtv/data')

for gtv_path in tqdm(list(source_dir.rglob("*gtv.nii.gz"))):
    gtv = nib.load(gtv_path)
    gtv_np = gtv.get_fdata()

    patient = gtv_path.name.split('_')[0]
    temporality = gtv_path.name.split('_')[1]
    target_gtv_dir = target_dir / patient / temporality
    target_gtv_dir.mkdir(parents=True, exist_ok=True)
    if len(np.unique(gtv_np)) != 2:
        gtv_np[(gtv_np == 1) | (gtv_np == 2)] = 1

    new_gtv = nib.Nifti1Image(gtv_np.astype(np.uint8), gtv.affine, gtv.header)
    nib.save(new_gtv, target_gtv_dir / gtv_path.name)

gen_gtv_dir = Path('./nnUNet_raw/CFB-GBM_gen/PredictionsTr')

for gtv in tqdm(gen_gtv_dir.glob("*.nii.gz")):
    id_patient_temporality = gtv.name.split('.')[0]
    patient = id_patient_temporality[:-1]
    temporality = id_patient_temporality[-1]
    target_gtv_dir = target_dir / patient / f't{temporality}'

    target_gtv_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(gtv, target_gtv_dir / f'{patient}_t{temporality}_gtv.nii.gz')

100%|██████████| 191/191 [03:13<00:00,  1.01s/it]
342it [00:01, 316.38it/s]


## Compute the RANO csv

In [1]:
from pathlib import Path
import nibabel as nib
import pandas as pd
from tqdm import tqdm
import numpy as np

root_dir = Path('./full_gtv/data')

def get_volume_size(nib_file):
        header = nib_file.header
        spacing = header.get_zooms()
        voxel_volume_mm3 = np.prod(spacing)
        data = nib_file.get_fdata()
        tumor_voxel_count = np.sum(data == 1)

        total_volume_mm3 = tumor_voxel_count * voxel_volume_mm3
        return total_volume_mm3 / 1000.0

def rano_metric(start_volume, end_volume, sequence_label):
    if start_volume is None or end_volume is None:
        return {f'reduction_rate_{sequence_label}': None, f'rano_{sequence_label}': None}

    RANO_class_label = {
        'CR': 'Complete Response (CR)',
        'PR': 'Partial Response (PR)',
        'SD': 'Stable Disease (SD)',
        'PD': 'Progressive Disease (PD)'
    }
    reduction_rate = (start_volume - end_volume) / start_volume
    if reduction_rate == 1:
        rano = RANO_class_label['CR']
    elif reduction_rate >= 0.65:
        rano = RANO_class_label['PR']
    elif reduction_rate >= -0.4:
        rano = RANO_class_label['SD']
    elif reduction_rate < -0.4:
        rano = RANO_class_label['PD']
    else:
        print(f'error reduction rate: {reduction_rate}')
    return {f'reduction_rate_{sequence_label}': round(reduction_rate, 4), f'rano_{sequence_label}': rano}

rows = []
for patient_path in tqdm(root_dir.iterdir()):
    if patient_path.is_dir():
        patient_id = patient_path.name

        gtv_t0_path = patient_path / 't0' / f'{patient_id}_t0_gtv.nii.gz'
        gtv_t0 = nib.load(gtv_t0_path) if gtv_t0_path.is_file() else None

        gtv_t1_path = patient_path / 't1' / f'{patient_id}_t1_gtv.nii.gz'
        gtv_t1 = nib.load(gtv_t1_path) if gtv_t1_path.is_file() else None

        gtv_t2_path = patient_path / 't2' / f'{patient_id}_t2_gtv.nii.gz'
        gtv_t2 = nib.load(gtv_t2_path) if gtv_t2_path.is_file() else None

        if gtv_t0 is None or (gtv_t1 is None and gtv_t2 is None):
            continue

        size_gtv_t0 = get_volume_size(gtv_t0) if gtv_t0 is not None else None
        size_gtv_t1 = get_volume_size(gtv_t1) if gtv_t1 is not None else None
        size_gtv_t2 = get_volume_size(gtv_t2) if gtv_t2 is not None else None

        rows.append({
            'id_patient': int(patient_id),
            'size_t0 (cm3)': round(size_gtv_t0, 2) if size_gtv_t0 is not None else None,
            'size_t1 (cm3)': round(size_gtv_t1, 2) if size_gtv_t1 is not None else None,
            'size_t2 (cm3)': round(size_gtv_t2, 2) if size_gtv_t2 is not None else None,
            **rano_metric(size_gtv_t0, size_gtv_t1, "t0_to_t1"),
            **rano_metric(size_gtv_t0, size_gtv_t2, "t0_to_t2"),
            **rano_metric(size_gtv_t1, size_gtv_t2, "t1_to_t2"),
        })

df_rano = pd.DataFrame(rows)

261it [01:20,  3.24it/s]


In [2]:
df_rano

,id_patient,size_t0 (cm3),size_t1 (cm3),size_t2 (cm3),reduction_rate_t0_to_t1,rano_t0_to_t1,reduction_rate_t0_to_t2,rano_t0_to_t2,reduction_rate_t1_to_t2,rano_t1_to_t2
0,101,52.78,71.00,86.38,-0.3453,Stable Disease (SD),-0.6367,Progressive Disease (PD),-0.2166,Stable Disease (SD)
1,103,16.35,33.60,NaN,-1.0555,Progressive Disease (PD),NaN,None,NaN,None
2,104,29.08,44.67,NaN,-0.5362,Progressive Disease (PD),NaN,None,NaN,None
3,105,27.07,39.45,50.90,-0.4572,Progressive Disease (PD),-0.8803,Progressive Disease (PD),-0.2903,Stable Disease (SD)
4,107,18.35,18.34,NaN,0.0001,Stable Disease (SD),NaN,None,NaN,None
...,...,...,...,...,...,...,...,...,...,...
160,93,9.26,14.79,0.19,-0.5978,Progressive Disease (PD),0.9791,Partial Response (PR),0.9869,Partial Response (PR)
161,94,42.60,66.41,89.62,-0.5589,Progressive Disease (PD),-1.1037,Progressive Disease (PD),-0.3495,Stable Disease (SD)
162,95,29.75,12.01,14.01,0.5963,Stable Disease (SD),0.5291,Stable Disease (SD),-0.1665,Stable Disease (SD)
163,97,60.23,12.77,3.26,0.7880,Partial Response (PR),0.9459,Partial Response (PR),0.7446,Partial Response (PR)


In [3]:
df_rano = df_rano.sort_values('id_patient')
df_rano

,id_patient,size_t0 (cm3),size_t1 (cm3),size_t2 (cm3),reduction_rate_t0_to_t1,rano_t0_to_t1,reduction_rate_t0_to_t2,rano_t0_to_t2,reduction_rate_t1_to_t2,rano_t1_to_t2
123,3,41.96,42.95,NaN,-0.0236,Stable Disease (SD),NaN,None,NaN,None
135,5,76.44,66.73,61.75,0.1271,Stable Disease (SD),0.1921,Stable Disease (SD),0.0745,Stable Disease (SD)
141,6,78.94,183.36,NaN,-1.3229,Progressive Disease (PD),NaN,None,NaN,None
145,7,53.03,30.86,NaN,0.4182,Stable Disease (SD),NaN,None,NaN,None
153,8,32.67,35.66,37.49,-0.0915,Stable Disease (SD),-0.1475,Stable Disease (SD),-0.0513,Stable Disease (SD)
...,...,...,...,...,...,...,...,...,...,...
117,261,33.80,11.05,8.27,0.6731,Partial Response (PR),0.7553,Partial Response (PR),0.2514,Stable Disease (SD)
118,262,44.47,17.03,11.35,0.6170,Stable Disease (SD),0.7448,Partial Response (PR),0.3337,Stable Disease (SD)
119,263,35.57,44.84,NaN,-0.2604,Stable Disease (SD),NaN,None,NaN,None
120,264,23.08,20.01,24.45,0.1329,Stable Disease (SD),-0.0598,Stable Disease (SD),-0.2222,Stable Disease (SD)


In [4]:
df_rano.to_csv('rano.csv', index=False)